# Skin Health Model - Colab Notebook

This notebook allows you to generate synthetic data, train the Skin Health Risk Prediction model, and make predictions directly in Google Colab.

In [ ]:
# 1. Install Dependencies (usually pre-installed in Colab, but good to be safe)
!pip install pandas numpy scikit-learn joblib

In [ ]:
# 2. Import Libraries
import pandas as pd
import numpy as np
import joblib
import random
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score

## 3. Data Generation
We will generate a synthetic dataset to train our model.

In [ ]:
# Configuration
NUM_SAMPLES = 2000
DATA_FILE = "skin_health_train.csv"

# Define Categories
SUGAR_INTAKE = ['Low', 'Medium', 'High']
DAIRY_INTAKE = ['None', 'Occasional', 'Daily']
PROCESSED_FOOD = ['Low', 'Medium', 'High']
SPICY_FOOD = ['Low', 'Medium', 'High']
DIET_TYPE = ['Veg', 'Non-Veg', 'Vegan']
OILY_FOOD = ['Low', 'Medium', 'High']
SKIN_TYPE = ['Dry', 'Normal', 'Oily']
EXERCISE = ['Yes', 'No']

NON_VEG_ITEMS = ['Chicken', 'Red Meat', 'Fish', 'Eggs']
VEG_ITEMS = ['Leafy Greens', 'Paneer/Dairy Rich', 'Starchy/Potato', 'Balanced Veg']
VEGAN_ITEMS = ['Leafy Greens', 'Starchy/Potato', 'Legumes/Pulses', 'Nuts/Seeds']

def generate_row():
    # 1. Randomized Inputs
    sugar = random.choice(SUGAR_INTAKE)
    dairy = random.choice(DAIRY_INTAKE)
    processed = random.choice(PROCESSED_FOOD)
    spicy = random.choice(SPICY_FOOD)
    diet_type = random.choice(DIET_TYPE)
    oily = random.choice(OILY_FOOD)
    
    if diet_type == 'Non-Veg':
        specific_food = random.choice(NON_VEG_ITEMS)
    elif diet_type == 'Veg':
        specific_food = random.choice(VEG_ITEMS)
    else: 
        specific_food = random.choice(VEGAN_ITEMS)
    
    # Feature Engineering
    if specific_food in ['Red Meat', 'Processed Meat']:
        food_category = 'High_Inflammation'
    elif specific_food in ['Starchy/Potato', 'Balanced Veg']:
        food_category = 'High_Glycemic'
    elif specific_food in ['Leafy Greens', 'Legumes/Pulses']:
        food_category = 'Antioxidant_Rich'
    elif specific_food in ['Fish', 'Nuts/Seeds']:
        food_category = 'Omega3_Rich'
    elif specific_food in ['Paneer/Dairy Rich', 'Eggs']:
        food_category = 'Hormonal_Trigger'
    else:
        food_category = 'Balanced'

    water = round(random.uniform(1.0, 4.0), 1)
    
    if random.random() < 0.4:
        sleep = random.choice([4, 5, 8, 9])
        stress = random.choice([1, 4, 5])
    else:
        sleep = random.randint(5, 8)
        stress = random.randint(2, 4)
        
    exercise = random.choice(EXERCISE)
    skin = random.choice(SKIN_TYPE)

    # 2. Risk Calculation
    risk_score = 0
    if sugar == 'High': risk_score += 2
    elif sugar == 'Medium': risk_score += 1
    if dairy == 'Daily': risk_score += 1
    if food_category == 'High_Inflammation': risk_score += 3
    if food_category == 'Hormonal_Trigger': risk_score += 2
    if food_category == 'High_Glycemic': risk_score += 1
    if food_category == 'Omega3_Rich': risk_score -= 1
    if food_category == 'Antioxidant_Rich': risk_score -= 1
    if processed == 'High': risk_score += 2
    elif processed == 'Medium': risk_score += 1
    if spicy == 'High': risk_score += 1
    if spicy == 'High' and skin == 'Oily': risk_score += 1 
    if diet_type == 'Non-Veg' and processed == 'High': risk_score += 1
    if oily == 'High': risk_score += 2
    elif oily == 'Medium': risk_score += 1
    if water < 1.5: risk_score += 1
    if sleep < 6: risk_score += 2
    elif sleep < 7: risk_score += 1
    if stress >= 4: risk_score += 2
    elif stress == 3: risk_score += 1
    if exercise == 'No': risk_score += 1
    if skin == 'Oily': risk_score += 2
    elif skin == 'Combination': risk_score += 1

    if risk_score >= 9: risk_label = 'High'
    elif risk_score >= 5: risk_label = 'Medium'
    else: risk_label = 'Low'

    return {
        "diet_type": diet_type,
        "specific_food_item": specific_food,
        "food_category": food_category,
        "sugar_intake": sugar,
        "dairy_intake": dairy,
        "processed_food_freq": processed,
        "spicy_food_freq": spicy,
        "oily_food_level": oily,
        "water_intake_liters": water,
        "sleep_hours": sleep,
        "stress_level": stress,
        "exercise": exercise,
        "skin_type": skin,
        "acne_risk": risk_label
    }

print(f"Generating {NUM_SAMPLES} samples...")
data = [generate_row() for _ in range(NUM_SAMPLES)]
df = pd.DataFrame(data)
df.to_csv(DATA_FILE, index=False)
print(f"Data generated and saved to {DATA_FILE}")
print(df['acne_risk'].value_counts())

## 4. Model Training
Now we train the Random Forest model using the generated data.

In [ ]:
# Load Data
df = pd.read_csv(DATA_FILE)
X = df.drop(columns=['acne_risk', 'specific_food_item'])
y = df['acne_risk']
X.fillna('None', inplace=True)

# Define Preprocessing
ordinal_cols = ['sugar_intake', 'dairy_intake', 'processed_food_freq', 
               'spicy_food_freq', 'oily_food_level', 'exercise']
ordinal_categories = [
    ['Low', 'Medium', 'High'], # sugar
    ['None', 'Occasional', 'Daily'], # dairy
    ['Low', 'Medium', 'High'], # processed
    ['Low', 'Medium', 'High'], # spicy
    ['Low', 'Medium', 'High'], # oily
    ['No', 'Yes'] # exercise
]
nominal_cols = ['diet_type', 'food_category', 'skin_type']
numeric_cols = ['water_intake_liters', 'sleep_hours', 'stress_level']

preprocessor = ColumnTransformer(
    transformers=[
        ('ord', OrdinalEncoder(categories=ordinal_categories), ordinal_cols),
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nominal_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

# Transform Data
X_processed = preprocessor.fit_transform(X)
feature_names = ordinal_cols + list(preprocessor.named_transformers_['nom'].get_feature_names_out(nominal_cols)) + numeric_cols

# Encode Target
target_le = LabelEncoder()
y_encoded = target_le.fit_transform(y)

# Split
X_train, X_test, y_train, y_test = train_test_split(X_processed, y_encoded, test_size=0.2, random_state=42)

# Train
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {acc*100:.2f}%")
print(classification_report(y_test, y_pred, target_names=target_le.classes_))

# Save Artifacts for Download
artifacts = {
    'preprocessor': preprocessor,
    'target_encoder': target_le,
    'feature_names': feature_names,
    'model': clf
}
joblib.dump(artifacts, 'skin_health_model_full.joblib')
print("Model saved as 'skin_health_model_full.joblib'")

## 5. Test Prediction
Test the model with a custom input.

In [ ]:
def predict_risk(input_data):
    # Convert input dict to DataFrame
    input_df = pd.DataFrame([input_data])
    
    # Preprocess
    X_new = preprocessor.transform(input_df)
    
    # Predict
    # Note: returns index (0, 1, 2)
    prediction_idx = clf.predict(X_new)[0]
    probabilities = clf.predict_proba(X_new)[0]
    
    # Decode label
    risk_label = target_le.inverse_transform([prediction_idx])[0]
    
    return risk_label, probabilities

# Example Input
sample_input = {
    'diet_type': 'Non-Veg',
    'food_category': 'High_Inflammation', # e.g. Red Meat
    'sugar_intake': 'High',
    'dairy_intake': 'Daily',
    'processed_food_freq': 'High',
    'spicy_food_freq': 'Medium',
    'oily_food_level': 'High',
    'exercise': 'No',
    'skin_type': 'Oily',
    'water_intake_liters': 1.0,
    'sleep_hours': 5,
    'stress_level': 5
}

# Run Prediction
risk, probs = predict_risk(sample_input)
print(f"Predicted Risk: {risk}")
print(f"Probabilities: {probs}") # Order matches target_le.classes_